In [ ]:
import functools
import json

import eradiate
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import mitsuba as mi
from eradiate.contexts import KernelContext
from eradiate.units import unit_registry as ureg
from eradiate.kernel import scene_parameter, UpdateParameter

eradiate.set_mode("mono")

In [ ]:
# Reindex land cover data
lc_map = xr.load_dataset("data/landcover_beijing.nc")["landcover_class"]
lc_values = np.unique(lc_map)

lc_indexes = xr.zeros_like(lc_map, dtype="int64")
lc_indexes.attrs = {}

for i, v in enumerate(lc_values):
    lc_indexes.values[lc_map == v] = i

lc_indexes.plot.imshow()

In [ ]:
# Declare materials
with open("data/colormap.json") as f:
    colormap = json.load(f)

colormap_indexes = {i: colormap[str(int(v))] for i, v in enumerate(lc_values)}

def material_reflectance(mat_id, ctx: KernelContext):
    mat_rgb = colormap_indexes[mat_id]
    w = ctx.si.w.m_as("nm")

    if w == 440.0:
        return float(mat_rgb[0]) / 255
    elif w == 550.0:
        return float(mat_rgb[1]) / 255
    elif w == 660.0:
        return float(mat_rgb[2]) / 255
    else:
        raise ValueError


reflectances = {
    i: functools.partial(material_reflectance, i) for i in range(len(lc_values))
}

In [ ]:
# Generate materials for kernel dictionary and scene parameters

def make_materials(reflectances):
    kdict = {}
    kpmap = {}

    for mat_id, reflectance in reflectances.items():
        kdict[f"00_material_lc_{mat_id}"] = {
            "type": "diffuse",
            "id": f"00_material_lc_{mat_id}",
            "reflectance": 0.5,
        }
        kpmap[f"00_material_lc_{mat_id}.reflectance.value"] = scene_parameter(
            reflectance,
            flags=UpdateParameter.Flags.SPECTRAL,
            node_type=mi.BSDF,
            node_id=f"00_material_lc_{mat_id}",
            parameter_relpath="reflectance.value",
        )

    kdict["01_material_elevation"] = {
        "type": "selectbsdf",
        "id": "01_material_elevation",
        "indices": {
            "type": "bitmap",
            "data": np.atleast_3d(lc_indexes.values).astype("float32"),
            "raw": True,
            "filter_type": "nearest",
            "wrap_mode": "clamp",
        },
        **{
            f"bsdf_lc_{mat_id}": {"type": "ref", "id": f"00_material_lc_{mat_id}"}
            for mat_id in reflectances.keys()
        },
    }

    return kdict, kpmap

In [ ]:
%reload_ext eradiate.notebook
# Generate Eradiate scene
kdict, kpmap = make_materials(reflectances)
kdict.update({
    "1_shape_elevation": {
        "type": "ply",
        "filename": "data/dem_beijing.ply",
        "face_normals": False,
        "bsdf": {"type": "ref", "id": "01_material_elevation"},
        "to_world": mi.ScalarTransform4f.scale([1, 1, 5]),
    }
})

exp = eradiate.experiments.AtmosphereExperiment(
    atmosphere=None,
    surface=None,
    measures={
        "type": "perspective",
        "origin": [-1000, -1000, 1000] * ureg.km,
        "target": [0, 0, 0],
        "up": [0, 0, 1],
        "fov": 5,
        "film_resolution": (320, 240),
        "srf": {"type": "delta", "wavelengths": [440, 550, 660]},
    },
    illumination={
        "type": "directional",
        "zenith": 30.0,
    },
    kdict=kdict,
    kpmap=kpmap,
)
exp.init(drop_parameters=False)

In [ ]:
result = eradiate.run(exp)

In [ ]:
da = result["radiance"].squeeze().transpose("y_index", "x_index", "w")
plt.imshow((da.values) ** (1.0 / 2.2))
plt.axis("off")

In [ ]:
mi_scene